In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import lightgbm as lgb
import time
from pathlib import Path

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
RANKER_DIR = PARQUET_DIR / "ranker"

print("loading ranker dataset...")
train_ranker = pd.read_parquet(RANKER_DIR / "train.parquet")
eval_ranker = pd.read_parquet(RANKER_DIR / "eval.parquet")
print(f"  train: {len(train_ranker):,} rows")
print(f"  eval:  {len(eval_ranker):,} rows")

ID_COLS = ["user_idx", "movie_idx"]
TARGET = "label"
FEATURE_COLS = [c for c in train_ranker.columns if c not in ID_COLS + [TARGET]]

X_train = train_ranker[FEATURE_COLS].values.astype(np.float32)
y_train = train_ranker[TARGET].values.astype(np.int8)
X_eval = eval_ranker[FEATURE_COLS].values.astype(np.float32)
y_eval = eval_ranker[TARGET].values.astype(np.int8)

del train_ranker, eval_ranker
import gc; gc.collect()

mb = (X_train.nbytes + X_eval.nbytes) / 1e6
print(f"\nmemory for X: {mb:.1f} mb")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")

train_set = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_COLS)
eval_set = lgb.Dataset(X_eval, label=y_eval, feature_name=FEATURE_COLS, reference=train_set)

params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "lambda_l2": 1.0,
    "verbose": -1,
    "num_threads": 1,
}

print("\ntraining lightgbm (single-threaded for macos stability)...")
t0 = time.time()
gbm = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, eval_set],
    valid_names=["train", "eval"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=20),
    ],
)
print(f"\ntraining done in {time.time()-t0:.1f}s")
print(f"best iteration: {gbm.best_iteration}")
print(f"best auc: train={gbm.best_score['train']['auc']:.4f}, eval={gbm.best_score['eval']['auc']:.4f}")

loading ranker dataset...
  train: 4,800,000 rows
  eval:  1,200,000 rows

memory for X: 480.0 mb
X_train shape: (4800000, 20), dtype: float32

training lightgbm (single-threaded for macos stability)...
Training until validation scores don't improve for 20 rounds
[20]	train's auc: 0.849052	eval's auc: 0.848686
[40]	train's auc: 0.853675	eval's auc: 0.85313
[60]	train's auc: 0.857269	eval's auc: 0.856651
[80]	train's auc: 0.861013	eval's auc: 0.860269
[100]	train's auc: 0.863801	eval's auc: 0.862995
[120]	train's auc: 0.86572	eval's auc: 0.864835
[140]	train's auc: 0.86712	eval's auc: 0.86616
[160]	train's auc: 0.868221	eval's auc: 0.867187
[180]	train's auc: 0.869201	eval's auc: 0.868104
[200]	train's auc: 0.870065	eval's auc: 0.868892
[220]	train's auc: 0.870833	eval's auc: 0.869583
[240]	train's auc: 0.871505	eval's auc: 0.870182
[260]	train's auc: 0.872124	eval's auc: 0.870725
[280]	train's auc: 0.87266	eval's auc: 0.871193
[300]	train's auc: 0.873169	eval's auc: 0.871632
[320]	trai

In [2]:
print("feature importance (gain — how much each feature reduced loss):")
print("=" * 60)
importances = gbm.feature_importance(importance_type="gain")
total = importances.sum()
sorted_pairs = sorted(zip(FEATURE_COLS, importances), key=lambda x: -x[1])
for name, gain in sorted_pairs:
    pct = 100 * gain / total
    bar = "█" * int(pct / 2)
    print(f"  {name:<22s} {pct:5.1f}%  {bar}")

print()
print("feature importance (split — how often the feature is used in splits):")
print("=" * 60)
splits = gbm.feature_importance(importance_type="split")
total_s = splits.sum()
sorted_s = sorted(zip(FEATURE_COLS, splits), key=lambda x: -x[1])
for name, n_splits in sorted_s:
    pct = 100 * n_splits / total_s
    print(f"  {name:<22s} {pct:5.1f}%  ({n_splits} splits)")

feature importance (gain — how much each feature reduced loss):
  tt_score                71.9%  ███████████████████████████████████
  m_num_ratings            6.9%  ███
  ug_total_ratings         4.5%  ██
  u_num_ratings            3.9%  █
  ug_pct_high              2.5%  █
  u_pct_high               2.0%  █
  m_pct_high               1.9%  
  ug_n_genres              1.7%  
  m_std_rating             0.8%  
  m_mean_rating            0.6%  
  m_pct_low                0.5%  
  u_active_seconds         0.5%  
  ug_mean_rating           0.4%  
  m_num_unique_users       0.4%  
  m_smoothed_mean          0.4%  
  u_min_rating             0.3%  
  u_mean_rating            0.3%  
  u_std_rating             0.3%  
  u_pct_low                0.1%  
  u_max_rating             0.0%  

feature importance (split — how often the feature is used in splits):
  m_num_ratings           12.4%  (3851 splits)
  ug_total_ratings        12.2%  (3787 splits)
  u_num_ratings           12.2%  (3777 splits)
 

In [6]:
import torch
import torch.nn as nn
import os
import math
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import faiss
faiss.omp_set_num_threads(1)
torch.set_num_threads(1)

CKPT_PATH = Path.home() / "projects" / "recsys" / "checkpoints" / "two_tower.pt"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)


# load two-tower
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
n_users = ckpt["n_users"]
n_movies = ckpt["n_movies"]
emb_dim = ckpt["config"]["emb_dim"]
user_to_idx = ckpt["user_to_idx"]
movie_to_idx = ckpt["movie_to_idx"]

tt_model = TwoTower(n_users, n_movies, dim=emb_dim).to(device)
tt_model.load_state_dict(ckpt["model_state_dict"])
tt_model.eval()
print(f"two-tower loaded: {n_users:,} × {n_movies:,} × {emb_dim}")

# item vectors + faiss
with torch.no_grad():
    item_vecs = tt_model.encode_item(torch.arange(n_movies, dtype=torch.long, device=device)).cpu().numpy().astype(np.float32)
index = faiss.IndexFlatIP(emb_dim)
index.add(item_vecs)
print(f"faiss index: {index.ntotal:,} items")

# feature tables for online ranker scoring
user_features = pd.read_parquet(PARQUET_DIR / "user_features.parquet")
movie_features = pd.read_parquet(PARQUET_DIR / "movie_features.parquet")
user_genre = pd.read_parquet(PARQUET_DIR / "user_genre_features.parquet")
movies = pd.read_parquet(PARQUET_DIR / "movies.parquet")

user_features["user_idx"] = user_features["userId"].map(user_to_idx)
user_features = user_features.dropna(subset=["user_idx"])
user_features["user_idx"] = user_features["user_idx"].astype(np.int32)

movie_features["movie_idx"] = movie_features["movieId"].map(movie_to_idx)
movie_features = movie_features.dropna(subset=["movie_idx"])
movie_features["movie_idx"] = movie_features["movie_idx"].astype(np.int32)

user_genre["user_idx"] = user_genre["userId"].map(user_to_idx)
user_genre = user_genre.dropna(subset=["user_idx"])
user_genre["user_idx"] = user_genre["user_idx"].astype(np.int32)

movies["movie_idx"] = movies["movieId"].map(movie_to_idx)
movies = movies.dropna(subset=["movie_idx"])
movies["movie_idx"] = movies["movie_idx"].astype(np.int32)

# build a (movie_idx -> list of genres) lookup for ug features
movies["genre_list"] = movies["genres"].str.split("|")
movie_to_genres = dict(zip(movies["movie_idx"].values, movies["genre_list"].values))

# build a per-user dict: {user_idx: {genre: (num_ratings, mean_rating, pct_high)}}
print("\nbuilding user_genre dict (one-time, ~10s)...")
t0 = time.time()
user_genre_dict = {}
for uidx, group in user_genre.groupby("user_idx"):
    user_genre_dict[int(uidx)] = {
        row["genre"]: (row["num_ratings"], row["mean_rating"], row["pct_high"])
        for _, row in group.iterrows()
    }
print(f"  done in {time.time()-t0:.1f}s, {len(user_genre_dict):,} users")

# global means for ug fallback
global_ug_mean = user_genre["mean_rating"].mean()
global_ug_pct_high = user_genre["pct_high"].mean()

# build raw ratings split same as before
ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df_raw = ratings[ratings["timestamp"] < cutoff_ts].copy()
val_df_raw = ratings[ratings["timestamp"] >= cutoff_ts].copy()
for d in (train_df_raw, val_df_raw):
    d["user_idx"] = d["userId"].map(user_to_idx)
    d["movie_idx"] = d["movieId"].map(movie_to_idx)
    d.dropna(subset=["user_idx", "movie_idx"], inplace=True)
    d["user_idx"] = d["user_idx"].astype(np.int32)
    d["movie_idx"] = d["movie_idx"].astype(np.int32)
print(f"train: {len(train_df_raw):,} | val: {len(val_df_raw):,}")

two-tower loaded: 150,330 × 45,058 × 64
faiss index: 45,058 items

building user_genre dict (one-time, ~10s)...
  done in 31.1s, 150,330 users
train: 22,454,535 | val: 454,141


In [7]:
import math
def _dcg(rels):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))


# precompute lookups for online feature building
user_feat_dict = user_features.set_index("user_idx")[
    ["num_ratings", "mean_rating", "std_rating", "min_rating", "max_rating",
     "active_seconds", "pct_high", "pct_low"]
].to_dict("index")

movie_feat_dict = movie_features.set_index("movie_idx")[
    ["num_ratings", "num_unique_users", "mean_rating", "std_rating",
     "pct_high", "pct_low", "smoothed_mean"]
].to_dict("index")

# val sampling
train_by_user = train_df_raw.groupby("user_idx")["movie_idx"].apply(set)
val_by_user_liked = (
    val_df_raw[val_df_raw["rating"] >= 4.0]
    .groupby("user_idx")["movie_idx"].apply(set)
)
eligible = list(val_by_user_liked.index)
print(f"eligible val users: {len(eligible):,}")

rng = np.random.RandomState(42)
sample_users = rng.choice(eligible, size=1000, replace=False).astype(np.int64)
print(f"sample size: {len(sample_users)}")

# popularity baseline
pop_score = np.zeros(n_movies, dtype=np.float32)
counts = train_df_raw["movie_idx"].value_counts()
pop_score[counts.index.values] = counts.values

K_OVERFETCH = 200

# encode all sample users in one batch
with torch.no_grad():
    sample_users_t = torch.tensor(sample_users, device=device)
    user_vecs_all = tt_model.encode_user(sample_users_t).cpu().numpy().astype(np.float32)

# faiss search: top-K_OVERFETCH per user
t_retrieve = time.time()
D_all, I_all = index.search(user_vecs_all, K_OVERFETCH)
t_retrieve = time.time() - t_retrieve
print(f"faiss retrieve done in {t_retrieve:.2f}s")


def build_features_batch(user_idx, candidate_movies, tt_scores):
    """build the 20-feature matrix for (user, candidate) pairs."""
    n = len(candidate_movies)
    uf = user_feat_dict.get(int(user_idx), None)
    user_genres_data = user_genre_dict.get(int(user_idx), {})
    
    # user features (constant across candidates)
    u_vals = [
        uf["num_ratings"], uf["mean_rating"], uf["std_rating"], uf["min_rating"],
        uf["max_rating"], uf["active_seconds"], uf["pct_high"], uf["pct_low"]
    ]
    
    feats = np.empty((n, len(FEATURE_COLS)), dtype=np.float32)
    
    for i, m_idx in enumerate(candidate_movies):
        m_idx = int(m_idx)
        mf = movie_feat_dict.get(m_idx, None)
        if mf is None:
            # unseen movie — shouldn't happen since candidates come from index
            feats[i] = 0.0
            continue
        
        # ug aggregates
        movie_genres = movie_to_genres.get(m_idx, [])
        movie_genres = [g for g in movie_genres if g != "(no genres listed)"]
        ug_n = 0
        ug_total = 0
        ug_mean_sum = 0.0
        ug_pct_sum = 0.0
        ug_counted = 0
        for g in movie_genres:
            stat = user_genres_data.get(g)
            if stat is not None:
                ug_n += 1
                ug_total += stat[0]
            ug_mean_sum += stat[1] if stat else global_ug_mean
            ug_pct_sum += stat[2] if stat else global_ug_pct_high
            ug_counted += 1
        ug_mean = ug_mean_sum / ug_counted if ug_counted else global_ug_mean
        ug_pct = ug_pct_sum / ug_counted if ug_counted else global_ug_pct_high
        
        # populate row in same order as FEATURE_COLS
        feats[i] = [
            uf["num_ratings"], uf["mean_rating"], uf["std_rating"], uf["min_rating"],
            uf["max_rating"], uf["active_seconds"], uf["pct_high"], uf["pct_low"],
            mf["num_ratings"], mf["num_unique_users"], mf["mean_rating"], mf["std_rating"],
            mf["pct_high"], mf["pct_low"], mf["smoothed_mean"],
            tt_scores[i],
            ug_n, ug_total, ug_mean, ug_pct,
        ]
    return feats


print(f"\nfeature col order: {FEATURE_COLS}")
print(f"running end-to-end eval (this is the slow cell — pure python loop, ~3-7 min)...")

metrics_pop = {f"recall@{k}": [] for k in (5, 10, 20)}
metrics_tt = {f"recall@{k}": [] for k in (5, 10, 20)}
metrics_full = {f"recall@{k}": [] for k in (5, 10, 20)}
metrics_pop.update({f"ndcg@{k}": [] for k in (5, 10, 20)})
metrics_tt.update({f"ndcg@{k}": [] for k in (5, 10, 20)})
metrics_full.update({f"ndcg@{k}": [] for k in (5, 10, 20)})

t0 = time.time()
for i, user_idx in enumerate(sample_users):
    seen = train_by_user.get(user_idx, set())
    liked = val_by_user_liked[user_idx]
    
    # ---- popularity ----
    pop_masked = pop_score.copy()
    pop_masked[list(seen)] = -np.inf
    top_pop = np.argpartition(-pop_masked, 20)[:20]
    top_pop = top_pop[np.argsort(-pop_masked[top_pop])]
    
    # ---- two-tower retrieval (top 200, mask seen, take top-20) ----
    raw_candidates = I_all[i]
    raw_scores = D_all[i]
    keep = ~np.isin(raw_candidates, list(seen))
    tt_candidates = raw_candidates[keep][:20]
    tt_scores_for_metrics = raw_scores[keep][:20]
    
    # ---- two-tower + ranker ----
    # take top-200 (after mask), build features, score with ranker, take top-20
    candidates_200 = raw_candidates[keep]
    scores_200 = raw_scores[keep]
    if len(candidates_200) < 20:
        # pad if not enough survivors
        full_candidates = candidates_200
        full_top20 = full_candidates
    else:
        feats = build_features_batch(user_idx, candidates_200, scores_200)
        ranker_scores = gbm.predict(feats)
        rank_order = np.argsort(-ranker_scores)
        full_top20 = candidates_200[rank_order[:20]]
    
    # compute metrics for all three
    for stack, top_arr, store in [
        ("pop", top_pop, metrics_pop),
        ("tt", tt_candidates, metrics_tt),
        ("full", full_top20, metrics_full),
    ]:
        for k in (5, 10, 20):
            top_k = top_arr[:k]
            hits = liked.intersection(top_k.tolist() if isinstance(top_k, np.ndarray) else top_k)
            store[f"recall@{k}"].append(len(hits) / len(liked))
            rels = [1 if mid in liked else 0 for mid in top_k]
            ideal = [1] * min(k, len(liked))
            ndcg = _dcg(rels) / _dcg(ideal) if ideal else 0
            store[f"ndcg@{k}"].append(ndcg)
    
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(sample_users)} users  {time.time()-t0:.1f}s")

print(f"\ndone in {time.time()-t0:.1f}s\n")

print(f"{'metric':<14s} {'popularity':>12s} {'tt-only':>10s} {'tt + ranker':>14s}")
print("-" * 54)
for k in (5, 10, 20):
    p = np.mean(metrics_pop[f"recall@{k}"])
    t = np.mean(metrics_tt[f"recall@{k}"])
    f = np.mean(metrics_full[f"recall@{k}"])
    print(f"recall@{k:<7d} {p:>12.4f} {t:>10.4f} {f:>14.4f}")
print()
for k in (5, 10, 20):
    p = np.mean(metrics_pop[f"ndcg@{k}"])
    t = np.mean(metrics_tt[f"ndcg@{k}"])
    f = np.mean(metrics_full[f"ndcg@{k}"])
    print(f"ndcg@{k:<9d} {p:>12.4f} {t:>10.4f} {f:>14.4f}")

eligible val users: 6,282
sample size: 1000
faiss retrieve done in 0.04s

feature col order: ['u_num_ratings', 'u_mean_rating', 'u_std_rating', 'u_min_rating', 'u_max_rating', 'u_active_seconds', 'u_pct_high', 'u_pct_low', 'm_num_ratings', 'm_num_unique_users', 'm_mean_rating', 'm_std_rating', 'm_pct_high', 'm_pct_low', 'm_smoothed_mean', 'tt_score', 'ug_n_genres', 'ug_total_ratings', 'ug_mean_rating', 'ug_pct_high']
running end-to-end eval (this is the slow cell — pure python loop, ~3-7 min)...
  100/1000 users  0.4s
  200/1000 users  0.7s
  300/1000 users  1.1s
  400/1000 users  1.5s
  500/1000 users  1.8s
  600/1000 users  2.2s
  700/1000 users  2.6s
  800/1000 users  2.9s
  900/1000 users  3.3s
  1000/1000 users  3.6s

done in 3.6s

metric           popularity    tt-only    tt + ranker
------------------------------------------------------
recall@5             0.0202     0.0258         0.0303
recall@10            0.0301     0.0435         0.0504
recall@20            0.0468     0.06

built a two-stage recommender on movielens-25m: two-tower retrieval (pop-weighted bpr negatives, faiss for ann lookup) + lightgbm ranker (20 features including the retrieval score, user/movie aggregates, and per-(user, genre) preferences). end-to-end recall@10 = 0.0504, a 67% lift over popularity baseline and 16% over retrieval-only. tt_score dominates ranker gain at 72%, with per-(user, genre) features contributing 9% — the strongest signal beyond retrieval is fine-grained content matching. eval latency: 3.6ms per user end-to-end.

In [8]:
RANKER_CKPT = Path.home() / "projects" / "recsys" / "checkpoints" / "ranker.lgb"
gbm.save_model(str(RANKER_CKPT), num_iteration=gbm.best_iteration)
print(f"saved ranker to: {RANKER_CKPT}")
print(f"size: {RANKER_CKPT.stat().st_size / 1e3:.1f} kb")

# write a summary json alongside it
import json
summary = {
    "n_features": len(FEATURE_COLS),
    "feature_cols": FEATURE_COLS,
    "best_iteration": gbm.best_iteration,
    "best_auc": {"train": float(gbm.best_score["train"]["auc"]),
                 "eval": float(gbm.best_score["eval"]["auc"])},
    "end_to_end_metrics": {
        "popularity": {f"recall@{k}": float(np.mean(metrics_pop[f"recall@{k}"])) for k in (5,10,20)},
        "tt_only":    {f"recall@{k}": float(np.mean(metrics_tt[f"recall@{k}"]))  for k in (5,10,20)},
        "tt_ranker":  {f"recall@{k}": float(np.mean(metrics_full[f"recall@{k}"])) for k in (5,10,20)},
    },
}
SUMMARY_PATH = Path.home() / "projects" / "recsys" / "checkpoints" / "ranker_summary.json"
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)
print(f"saved summary to: {SUMMARY_PATH}")

saved ranker to: /Users/nitishpatil/projects/recsys/checkpoints/ranker.lgb
size: 3567.3 kb
saved summary to: /Users/nitishpatil/projects/recsys/checkpoints/ranker_summary.json
